## Q1: Roles of Driver, Cluster Manager, and Executor
Driver

The Driver is the main program that coordinates the Spark application. It:

Creates the SparkSession/SparkContext.
Converts transformations into a DAG.
Schedules tasks.
Communicates with executors.
Cluster Manager

The Cluster Manager manages cluster resources. It allocates CPU and memory to the Spark application.

Examples include:

Standalone
YARN
Kubernetes
Executor

An Executor runs tasks on worker nodes and stores data in memory or on disk for caching.

Driver
   ↓
Cluster Manager
   ↓
Executors on Worker Nodes

## Q2: How does Lazy Evaluation improve performance?

Spark does not immediately execute transformations such as:
df.filter(...)
df.select(...)
df.groupBy(...)

Instead, Spark builds a DAG (Directed Acyclic Graph) of operations.

Execution begins only when an action such as show(), count(), or write() is called.

This allows Spark to:

Combine multiple operations.
Optimize the execution plan.
Avoid unnecessary computations.
Reduce data movement and disk I/O.

result = (
    df.filter(df["price"] > 100)
      .select("product_id", "price")
)

result.show()

The operations are optimized and executed together only when show() is called.

In [1]:
import sys

print(sys.executable)
print(sys.version)

c:\Users\agarw\Desktop\Celebal\Week 6_Spark Questions 2\.venv\Scripts\python.exe
3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Week6") \
    .master("local[*]") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 4.2.0


## Q3: Reading a CSV File

In [3]:
df = spark.read.csv(
    "C:\\Users\\agarw\\Desktop\\Celebal\\Week 6_Spark Questions 2\\data\\source.csv",
    header=True,
    inferSchema=True
)

df.show()
df.printSchema()

+----------+-----+-----------+---------+------+------+--------+-------+----------+--------+
|product_id|price|   category|   status|amount|region|priority|user_id|base_price|old_name|
+----------+-----+-----------+---------+------+------+--------+-------+----------+--------+
|      P001| 1200|Electronics|Completed|  1500| North|    High|   U001|      1200| Value A|
|      P002|  500|  Furniture|  Pending|   500| South|     Low|   U002|       500| Value B|
|      P003| 2500|Electronics|Completed|  2500| North|  Medium|   U003|      2500| Value C|
|      P004|  800|   Clothing|Completed|   800|  West|    High|   NULL|       800| Value D|
|      P005| 1500|Electronics|Completed|  1800|  East|     Low|   U005|      1500| Value E|
|      P006|  300|Electronics|  Pending|   300| North|    High|   U006|       300| Value F|
|      P007| 2200|  Furniture|Completed|  2200|  West|  Medium|   U007|      2200| Value G|
|      P008|  950|Electronics|Completed|  1200| North|    High|   U008|       95

## Q4: CSV vs Parquet

CSV is a row-based text format, while Parquet is a columnar binary storage format.

CSV files are easy to read and share but usually require more storage and processing time. Parquet stores data by columns, allowing Spark to read only the columns required by a query.

For example, if a query needs only product_id and price, Parquet can read only those columns. This reduces disk I/O, network transfer, and memory usage, resulting in better performance for analytical workloads.

## Q5: Selecting Electronics Products

In [4]:
from pyspark.sql.functions import col

result_q5 = df.filter(
    col("category") == "Electronics"
).select(
    "product_id",
    "price"
)

result_q5.show()

+----------+-----+
|product_id|price|
+----------+-----+
|      P001| 1200|
|      P003| 2500|
|      P005| 1500|
|      P006|  300|
|      P008|  950|
|      P010| 1800|
+----------+-----+



## Q6: Renaming and Casting Columns

In [5]:
result_q6 = (
    df
    .withColumnRenamed("old_name", "new_name")
    .withColumn(
        "price",
        col("price").cast("double")
    )
)

result_q6.printSchema()
result_q6.show()

root
 |-- product_id: string (nullable = true)
 |-- price: double (nullable = true)
 |-- category: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- base_price: integer (nullable = true)
 |-- new_name: string (nullable = true)

+----------+------+-----------+---------+------+------+--------+-------+----------+--------+
|product_id| price|   category|   status|amount|region|priority|user_id|base_price|new_name|
+----------+------+-----------+---------+------+------+--------+-------+----------+--------+
|      P001|1200.0|Electronics|Completed|  1500| North|    High|   U001|      1200| Value A|
|      P002| 500.0|  Furniture|  Pending|   500| South|     Low|   U002|       500| Value B|
|      P003|2500.0|Electronics|Completed|  2500| North|  Medium|   U003|      2500| Value C|
|      P004| 800.0|   Clothing|Completed

## Q7: DAG and Fault Tolerance

Spark maintains a Lineage Graph, also called a DAG, that records the sequence of transformations used to create data.

If a worker node fails and a partition is lost, Spark uses the lineage information to recompute only the lost partition from the original data and transformations.

Therefore, Spark does not need to restart the entire application, providing fault tolerance.

## Q8: Filtering Completed Orders

In [6]:
result_q8 = df.filter(
    (col("status") == "Completed") &
    (col("amount") > 1000)
)

result_q8.show()

+----------+-----+-----------+---------+------+------+--------+-------+----------+--------+
|product_id|price|   category|   status|amount|region|priority|user_id|base_price|old_name|
+----------+-----+-----------+---------+------+------+--------+-------+----------+--------+
|      P001| 1200|Electronics|Completed|  1500| North|    High|   U001|      1200| Value A|
|      P003| 2500|Electronics|Completed|  2500| North|  Medium|   U003|      2500| Value C|
|      P005| 1500|Electronics|Completed|  1800|  East|     Low|   U005|      1500| Value E|
|      P007| 2200|  Furniture|Completed|  2200|  West|  Medium|   U007|      2200| Value G|
|      P008|  950|Electronics|Completed|  1200| North|    High|   U008|       950| Value H|
|      P010| 1800|Electronics|Completed|  2000| North|    High|   U010|      1800| Value J|
+----------+-----+-----------+---------+------+------+--------+-------+----------+--------+



## Q9: Predicate Pushdown

Predicate Pushdown is an optimization technique in which Spark pushes filter conditions closer to the data source.

For example, when reading a Parquet file and filtering rows where amount is greater than 1000, Spark can apply the filter while reading the data instead of loading the entire dataset first.

This reduces the amount of data read from storage and loaded into memory. As a result, disk I/O, network traffic, memory usage, and processing time are reduced.

## Q10: Calculating Final Price

In [7]:
result_q10 = df.withColumn(
    "final_price",
    col("base_price") * 1.18
)

result_q10.select(
    "product_id",
    "base_price",
    "final_price"
).show()

+----------+----------+-----------+
|product_id|base_price|final_price|
+----------+----------+-----------+
|      P001|      1200|     1416.0|
|      P002|       500|      590.0|
|      P003|      2500|     2950.0|
|      P004|       800|      944.0|
|      P005|      1500|     1770.0|
|      P006|       300|      354.0|
|      P007|      2200|     2596.0|
|      P008|       950|     1121.0|
|      P009|       700|      826.0|
|      P010|      1800|     2124.0|
+----------+----------+-----------+



## Q11: Transformations and Actions

Transformations create a new DataFrame or RDD and are lazily evaluated. They do not immediately execute the computation.

Examples of transformations:
1. filter()
2. select()

Other examples include groupBy(), join(), and withColumn().

Actions trigger the actual execution of Spark computations.

Examples of actions:
1. show()
2. count()

Other examples include collect(), first(), and write().

In [19]:
import os

os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["hadoop.home.dir"] = "C:\\hadoop"

In [20]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Week6") \
    .master("local[*]") \
    .getOrCreate()

## Q12: Parquet to CSV Processing Pipeline

In [33]:
from pyspark.sql.functions import col

df = spark.read.csv(
    "data/source.csv",
    header=True,
    inferSchema=True
)

result_q12 = df.filter(
    col("user_id").isNotNull()
)

result_q12.show()

# Output pipeline:
# result_q12.write \
#     .mode("overwrite") \
#     .option("header", True) \
#     .csv("data/output_csv")

+----------+-----+-----------+---------+------+------+--------+-------+----------+--------+
|product_id|price|   category|   status|amount|region|priority|user_id|base_price|old_name|
+----------+-----+-----------+---------+------+------+--------+-------+----------+--------+
|      P001| 1200|Electronics|Completed|  1500| North|    High|   U001|      1200| Value A|
|      P002|  500|  Furniture|  Pending|   500| South|     Low|   U002|       500| Value B|
|      P003| 2500|Electronics|Completed|  2500| North|  Medium|   U003|      2500| Value C|
|      P005| 1500|Electronics|Completed|  1800|  East|     Low|   U005|      1500| Value E|
|      P006|  300|Electronics|  Pending|   300| North|    High|   U006|       300| Value F|
|      P007| 2200|  Furniture|Completed|  2200|  West|  Medium|   U007|      2200| Value G|
|      P008|  950|Electronics|Completed|  1200| North|    High|   U008|       950| Value H|
|      P009|  700|   Clothing|  Pending|   700| South|     Low|   U009|       70

This pipeline reads the dataset, filters out records where user_id is null, and writes the cleaned result to a CSV output directory. The original question specifies Parquet as the input format; however, due to the Windows Hadoop file-system configuration, CSV was used as the practical input format for demonstration.

## Q13: Client Mode vs Cluster Mode

In Client Mode, the Driver runs on the machine from which the Spark application is submitted. This mode is useful for interactive development and debugging.

In Cluster Mode, the Driver runs inside the cluster. This mode is more suitable for production applications and long-running jobs.

The main difference is the location of the Driver program.

## Q14: Filtering by Region or Priority

In [35]:
result_q14 = df.filter(
    (col("region") == "North") |
    (col("priority") == "High")
)

result_q14.show()

+----------+-----+-----------+---------+------+------+--------+-------+----------+--------+
|product_id|price|   category|   status|amount|region|priority|user_id|base_price|old_name|
+----------+-----+-----------+---------+------+------+--------+-------+----------+--------+
|      P001| 1200|Electronics|Completed|  1500| North|    High|   U001|      1200| Value A|
|      P003| 2500|Electronics|Completed|  2500| North|  Medium|   U003|      2500| Value C|
|      P004|  800|   Clothing|Completed|   800|  West|    High|   NULL|       800| Value D|
|      P006|  300|Electronics|  Pending|   300| North|    High|   U006|       300| Value F|
|      P008|  950|Electronics|Completed|  1200| North|    High|   U008|       950| Value H|
|      P010| 1800|Electronics|Completed|  2000| North|    High|   U010|      1800| Value J|
+----------+-----+-----------+---------+------+------+--------+-------+----------+--------+



## Q15: show(5) vs collect()

show(5) displays only five rows from the DataFrame and is safer when exploring a very large dataset.

collect() retrieves all rows and brings them to the Driver node. For a multi-terabyte dataset, this can consume all available Driver memory and cause an OutOfMemoryError.

Therefore, show(5) is safer because it limits the amount of data returned and avoids loading the entire dataset into the Driver's memory.

Week6_Spark_Questions.ipynb
│
├── Q1 - Theory
├── Q2 - Theory
├── Q3 - CSV Read Code
├── Q4 - CSV vs Parquet
├── Q5 - Filter and Select Code
├── Q6 - Rename and Cast Code
├── Q7 - DAG and Fault Tolerance
├── Q8 - Filter Orders Code
├── Q9 - Predicate Pushdown
├── Q10 - Add Column Code
├── Q11 - Transformations vs Actions
├── Q12 - Parquet to CSV Code
├── Q13 - Client vs Cluster Mode
├── Q14 - Filter with OR Code
└── Q15 - show(5) vs collect()

## Final Insights

### Architecture
Spark uses a Driver to coordinate the application, a Cluster Manager to allocate resources, and Executors to perform tasks on worker nodes.

### Performance
Spark improves performance through Lazy Evaluation, DAG optimization, in-memory processing, Predicate Pushdown, and efficient columnar formats such as Parquet.

### Data Processing
The assignment demonstrates a complete data processing workflow involving reading data, transforming DataFrames, filtering records, handling null values, and writing processed results.

### Best Practices
For large datasets, `show()` or `limit()` should be preferred for exploration instead of `collect()`, which can overload the Driver's memory.